# Session 4 — Glacier du Mont Miné: modelling deglaciation dates

*This session is taught within the course "Dates and Rates of landscape evolution" (G. King and C. Schmidt). It builds on Sessions 1–3; if you did not attend them, go through the first IGM runs of Session 2 (`session2/1-lecture-igm.md`, `session2/first-runs/`) first.*

Run this notebook in your RenkuLab session, from `session4/mont-mine/`.

We model the evolution of [Glacier du Mont Miné](https://map.geo.admin.ch/#/map?lang=en&center=2607500,1100500&z=6) (Val d'Hérens, Valais) from the Little Ice Age (LIA, ~1850) to today. The goal is to **calibrate the glacier model against the ages of rock samples** collected in the glacier forefield: for each sample, the model gives the year when the site became ice-free, to be compared with the measured age.

The retreat of the Mont Miné since the LIA is illustrated on slides 28–30 of [this presentation](https://www.unige.ch/forel/application/files/3116/5476/6042/F-Evolene-Paysagesglaciaires_compressed.pdf).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../utils"))   # course plotting helpers
from helper import *

RGI_ID = "RGI2000-v7.0-G-11-00907"   # Glacier du Mont Miné
SAMPLES = "Samples_2024.csv"          # sample sites (ID, CH1903 x/y, UTM32 x/y)

## 1. The model set-up

`experiment/params.yaml`:
- downloads the data for the glacier with `oggm_shop` (topography, ice thickness, outline);
- runs from 1700 to 2024, starting from **today's geometry**, with a surface mass balance (SMB) that depends on elevation and on the equilibrium line altitude (ELA): `smb: method: simple`;
- the ELA is lowered first (advance to the LIA maximum, ~1850), then raised (retreat until today).

In [ ]:
!cat experiment/params.yaml

The SMB is a piecewise linear function of elevation: gradient $\beta_\mathrm{abl}$ (`gradabl`) below the ELA, $\beta_\mathrm{acc}$ (`gradacc`) above, capped at $b_\mathrm{max}$ (`accmax`). Each row of the array gives these values at a given time; they are interpolated linearly in between.

In [ ]:
plot_smb_simple(ela=2900, gradabl=0.009, gradacc=0.005, accmax=2.0, zmin=1800, zmax=3800)

## 2. Run the model

The first run downloads the data (≈ 1 min). The run takes a few minutes.

In [ ]:
!igm_run +experiment=params hydra.run.dir=outputs/run_v1

In [ ]:
run = "outputs/run_v1"
show_dashboard(run, n=4)
analyze_time_series(os.path.join(run, "output_ts.nc"))

## 3. Ice cover at the sample sites

In [ ]:
plot_samples_map(f"data/input_{RGI_ID}.nc", SAMPLES)          # today's glacier and the sample sites
plot_samples_map(os.path.join(run, "output.nc"), SAMPLES, year=1850)   # modelled LIA glacier

In [ ]:
animate_glacier_evolution(os.path.join(run, "output.nc"), variable="thk")

The next plot shows the modelled ice thickness through time at each sample site, and the **modelled deglaciation year** (last time the site is covered by ice). Enter the measured ages in `ages` (calendar years, or years before 2024) to compare.

In [ ]:
ages = {}   # e.g. {"24FER1": 1890, "24FER2": 1925, ...}  <- measured ages
table = plot_thk_at_samples(os.path.join(run, "output.nc"), SAMPLES, ages=ages)
table

## 4. Calibration — your turn!

Copy `experiment/params.yaml` (e.g. `params_v2.yaml`), change the parameters, run into a new folder (`hydra.run.dir=outputs/run_v2`), and compare the modelled deglaciation years with the measured ages:

1. **ELA history**: first adjust the ELA values in time (when and how low is the LIA maximum? how fast is the rise afterwards?).
2. **SMB gradients and maximum accumulation** (`gradabl`, `gradacc`, `accmax`).
3. **Basal sliding**: `processes.iceflow.physics.sliding.tau_ref` — e.g. 0.4 (less sliding) and 0.1 (more sliding). Note: last year's set-up (IGM 3.0) used a stickier bed (`init_slidingco: 0.25`, equivalent to `tau_ref` ≈ 1.2 MPa).

Questions:
- Which parameters control the *timing* of deglaciation at the sites, and which control the *extent* of the LIA glacier?
- Can you match all samples with a single ELA history? If not, why (model, data, sample interpretation)?

In [ ]:
# !igm_run +experiment=params_v2 hydra.run.dir=outputs/run_v2
# plot_thk_at_samples("outputs/run_v2/output.nc", SAMPLES, ages=ages)

## To go further: a climate-driven SMB

Instead of prescribing the ELA, use the climate-driven SMB of the mini-project (`climate` + `smb` with `method: oggm`, see `project/my-glacier/experiment/params_past.yaml`). The observed climate is used from 1901; before 1901, the pre-industrial cooling is set with `delta_temp` in `clim_trend_array`.

`Samples_XXXX.csv` contains a second set of sample sites (IDs `19FER...`), in the same format (last two columns = UTM coordinates): `plot_thk_at_samples(..., "Samples_XXXX.csv")`.